In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time

USERNAME = '***REMOVED***'
PASSWORD = '***REMOVED***'
TEST_URL = 'https://www.instagram.com/p/C8T3xf2s9Ec/'  # ← 크롤링할 게시물 주소

options = Options()
options.add_experimental_option("detach", True)
options.add_experimental_option("excludeSwitches", ["enable-logging"])
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})

# 로그인
driver.get('https://www.instagram.com/accounts/login/')
WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.NAME, "username")))
driver.find_element(By.NAME, 'username').send_keys(USERNAME)
driver.find_element(By.NAME, 'password').send_keys(PASSWORD)
driver.find_element(By.NAME, 'password').send_keys(Keys.ENTER)
print("✅ 로그인 완료")

time.sleep(5)
try:
    WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.XPATH, "//button[text()='나중에 하기']"))
    ).click()
    print("✅ 팝업 닫기 완료")
except:
    print("ℹ️ 팝업 없음")

# 게시물 열기
driver.get(TEST_URL)
time.sleep(3)

# 본문 전체 텍스트 추출
try:
    content_div = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located(
            (By.XPATH, '//section/main/div/div[1]/div/div[2]')
        )
    )
    content = content_div.text
except Exception as e:
    content = f"❌ 본문 추출 실패 → {e}"

# 날짜 추출 (기존 그대로 유지)
try:
    date = driver.find_element(By.TAG_NAME, 'time').get_attribute('datetime')
except:
    date = "날짜 추출 실패"

# 결과 출력
print("\n📌 크롤링 결과")
print(f"본문:\n{content}")
print(f"날짜: {date}")


✅ 로그인 완료
ℹ️ 팝업 없음

📌 크롤링 결과
본문:
seoul_trends
•
팔로우
seoul_trends
 10주
얼마 남지않은 황금연휴!
설 연휴 전에 알아두면 좋을 꿀정보 총정리❤
특히 설에 고향가는 사람들이
알아두면 좋을 듯!
@@이거 봐봐
.
.
📌더 많은 데이트코스 정보는?
@seoul_trends
#설연휴 #설날 #설연휴그램 #고속도로 #차례상 #임시공휴일 #ktx예매
munjaehyi457
 10주
@ljcslove
좋아요 1개
답글 달기
rmflawk76
 10주
@ai5013
좋아요 1개
답글 달기
leeminhye423
 10주
@dongho0331
좋아요 1개
답글 달기
geonyong_eom
 10주
@yu_naang
좋아요 1개
답글 달기
un7840679
 10주
@chajiyoung66
좋아요 1개
답글 달기
lkong3193
 10주
@ltth_pink
좋아요 1개
답글 달기
ototittt
 10주
@zzzaaaaaang
좋아요 1개
답글 달기
좋아요 766개
1월 13일
날짜: 2025-01-13T02:54:40.000Z


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import random
import os

USERNAME = '***REMOVED***'
PASSWORD = '***REMOVED***'

# 1. 크롬 설정
options = Options()
options.add_experimental_option("detach", True)
options.add_experimental_option("excludeSwitches", ["enable-logging"])
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})

# 2. 로그인
driver.get('https://www.instagram.com/accounts/login/')
WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.NAME, "username")))
driver.find_element(By.NAME, 'username').send_keys(USERNAME)
driver.find_element(By.NAME, 'password').send_keys(PASSWORD)
driver.find_element(By.NAME, 'password').send_keys(Keys.ENTER)
print("✅ 로그인 완료")

# 3. 팝업 닫기
time.sleep(5)
try:
    WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.XPATH, "//button[text()='나중에 하기']"))
    ).click()
    print("✅ 팝업 닫기 완료")
except:
    print("ℹ️ 팝업 없음")

# 4. 링크 불러오기
df_links = pd.read_csv("임시공휴일_링크2.csv", encoding='utf-8-sig')
links = df_links[df_links.columns[0]].dropna().tolist()
print(f"🔗 총 링크 수: {len(links)}")

# 5. 본문 추출 함수
def extract_post_content(driver, url):
    try:
        driver.get(url)

        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(2)  # 렌더링 여유

        try:
            content_div = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.XPATH, '//section/main/div/div[1]/div/div[2]'))
            )
            content = content_div.text
        except:
            content = "본문 없음"

        try:
            date = driver.find_element(By.TAG_NAME, 'time').get_attribute('datetime')
        except:
            date = "날짜 없음"

        return {"링크": url, "본문내용": content, "날짜": date}

    except Exception as e:
        print(f"❌ 실패: {url} ➡ {e}")
        return {"링크": url, "본문내용": "", "날짜": ""}

# 6. 파일 초기화 및 헤더 작성
output_file = "임시공휴일링크2_본문_크롤링결과.csv"
with open(output_file, mode='w', encoding='utf-8-sig', newline='') as f:
    df_header = pd.DataFrame(columns=["링크", "본문내용", "날짜"])
    df_header.to_csv(f, index=False)

# 7. 반복 크롤링 및 저장
for idx, link in enumerate(links, 1):
    print(f"📥 ({idx}/{len(links)}) 크롤링 중: {link}")
    data = extract_post_content(driver, link)
    df_row = pd.DataFrame([data])
    
    # 성공한 데이터만 바로 저장
    with open(output_file, mode='a', encoding='utf-8-sig', newline='') as f:
        df_row.to_csv(f, index=False, header=False)
    
    time.sleep(random.uniform(2, 4))  # 크롤링 속도 완화

print("✅ 전체 크롤링 완료 및 CSV 저장 완료!")


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import random

USERNAME = '***REMOVED***'
PASSWORD = '***REMOVED***'

# 1. 크롬 설정
options = Options()
options.add_experimental_option("detach", True)
options.add_experimental_option("excludeSwitches", ["enable-logging"])
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
})

# 2. 로그인
driver.get('https://www.instagram.com/accounts/login/')
WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.NAME, "username")))
driver.find_element(By.NAME, 'username').send_keys(USERNAME)
driver.find_element(By.NAME, 'password').send_keys(PASSWORD)
driver.find_element(By.NAME, 'password').send_keys(Keys.ENTER)
print("✅ 로그인 완료")

# 3. 팝업 닫기
time.sleep(5)
try:
    WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.XPATH, "//button[text()='나중에 하기']"))
    ).click()
    print("✅ 팝업 닫기 완료")
except:
    print("ℹ️ 팝업 없음")

# 4. 링크 불러오기
df_links = pd.read_csv("임시공휴일_링크2.csv", encoding='utf-8-sig')
links = df_links[df_links.columns[0]].dropna().tolist()
print(f"🔗 총 링크 수: {len(links)}")

# 5. 본문 추출 함수
def extract_post_content(driver, url):
    try:
        driver.get(url)

        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(2)  # 렌더링 여유

        # 실제 본문 추출
        try:
            content_div = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.XPATH, '//section/main/div/div[1]/div/div[2]'))
            )
            content = content_div.text
        except:
            content = "본문 없음"

        try:
            date = driver.find_element(By.TAG_NAME, 'time').get_attribute('datetime')
        except:
            date = "날짜 없음"

        return {"링크": url, "본문내용": content, "날짜": date}

    except Exception as e:
        print(f"❌ 실패: {url} ➡ {e}")
        return {"링크": url, "본문내용": "", "날짜": ""}

# 6. 반복 크롤링 실행
post_data = []
for idx, link in enumerate(links, 1):
    print(f"📥 ({idx}/{len(links)}) 크롤링 중: {link}")
    data = extract_post_content(driver, link)
    post_data.append(data)
    time.sleep(random.uniform(2, 4))  # 크롤링 속도 완화

# 7. 결과 저장
df = pd.DataFrame(post_data)
df.to_csv("임시공휴일링크3_본문_크롤링결과.csv", index=False, encoding='utf-8-sig')
print("✅ 크롤링 완료 및 CSV 저장!")